### Extract and initial clean

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/online_retail_II.csv")
print(f"Raw rows: {len(df):,}")

# Standardize column names now, to avoid the "Customer ID" space issue everywhere downstream
df = df.rename(columns={"Customer ID": "customer_id_raw"})
df.columns = [c.lower() for c in df.columns]

print(df.columns.tolist())

Raw rows: 1,067,371
['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id_raw', 'country']


### Apply the exclusion rule from your schema design (zero/negative price = stock adjustments, not sales)

In [2]:
before = len(df)
df = df[df["price"] > 0].copy()
print(f"Excluded {before - len(df):,} rows with zero/negative price (stock adjustments, not real sales)")
print(f"Remaining rows: {len(df):,}")

Excluded 6,207 rows with zero/negative price (stock adjustments, not real sales)
Remaining rows: 1,061,164


### Derive is_return (from quantity, not Invoice prefix, per your documented finding)

In [3]:
df["is_return"] = df["quantity"] < 0
print(df["is_return"].value_counts())

is_return
False    1041671
True       19493
Name: count, dtype: int64


### Build dim_date

In [4]:
df["invoicedate"] = pd.to_datetime(df["invoicedate"])

date_range = pd.date_range(start=df["invoicedate"].min().normalize(), end=df["invoicedate"].max().normalize(), freq="D")

dim_date = pd.DataFrame({"full_date": date_range})
dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["day_of_week"] = dim_date["full_date"].dt.day_name()
dim_date["is_weekend"] = dim_date["full_date"].dt.dayofweek >= 5

dim_date = dim_date[["date_key", "full_date", "year", "quarter", "month", "month_name", "day_of_week", "is_weekend"]]
print(f"dim_date rows: {len(dim_date)}")
dim_date.head()

dim_date rows: 739


,date_key,full_date,year,quarter,month,month_name,day_of_week,is_weekend
0,20091201,2009-12-01,2009,4,12,December,Tuesday,False
1,20091202,2009-12-02,2009,4,12,December,Wednesday,False
2,20091203,2009-12-03,2009,4,12,December,Thursday,False
3,20091204,2009-12-04,2009,4,12,December,Friday,False
4,20091205,2009-12-05,2009,4,12,December,Saturday,True


### Build dim_country (with your UK-dominant region grouping)

In [5]:
def assign_region(country):
    if country == "United Kingdom":
        return "United Kingdom"
    elif country in ["EIRE", "Channel Islands"]:
        return "UK & Ireland"
    elif country == "Unspecified":
        return "Unknown"
    else:
        return "Europe" if country in [
            "Germany", "France", "Netherlands", "Spain", "Switzerland", "Belgium",
            "Portugal", "Italy", "Norway", "Sweden", "Cyprus", "Finland", "Austria",
            "Denmark", "Greece", "Poland", "Malta", "Lithuania", "Lebanon"
        ] else "Rest of World"

unique_countries = df["country"].dropna().unique()
dim_country = pd.DataFrame({"country_name": unique_countries})
dim_country["region"] = dim_country["country_name"].apply(assign_region)
dim_country["country_key"] = range(1, len(dim_country) + 1)
dim_country = dim_country[["country_key", "country_name", "region"]]

print(dim_country["region"].value_counts())
dim_country.head(10)

region
Rest of World     20
Europe            19
UK & Ireland       2
United Kingdom     1
Unknown            1
Name: count, dtype: int64


,country_key,country_name,region
0,1,United Kingdom,United Kingdom
1,2,France,Europe
2,3,USA,Rest of World
3,4,Belgium,Europe
4,5,Australia,Rest of World
5,6,EIRE,UK & Ireland
6,7,Germany,Europe
7,8,Portugal,Europe
8,9,Japan,Rest of World
9,10,Denmark,Europe


### Build dim_product (category via your keyword dictionary, plus the Unknown Product placeholder)

In [6]:
category_keywords = {
    "Christmas & Seasonal": ["CHRISTMAS", "XMAS", "EASTER", "HALLOWEEN"],
    "Lighting": ["LIGHT", "LAMP", "CANDLE", "T-LIGHT"],
    "Kitchen & Dining": ["CAKE", "TEA", "GLASS", "BOTTLE", "MUG", "CUP", "LUNCH"],
    "Bags & Storage": ["BAG", "BOX", "TIN", "BASKET"],
    "Stationery & Paper": ["CARD", "PAPER", "NOTEBOOK", "PENCIL"],
    "Home Decor": ["HEART", "SIGN", "HANGING", "DECORATION", "HOLDER"],
}

def assign_category(description):
    if pd.isna(description):
        return "Other"
    desc_upper = str(description).upper()
    for category, keywords in category_keywords.items():
        if any(kw in desc_upper for kw in keywords):
            return category
    return "Other"

# One row per stock_code (a product might have slightly varying descriptions across rows — take the most common one)
product_desc = df.groupby("stockcode")["description"].agg(lambda x: x.mode()[0] if not x.mode().empty else "UNKNOWN").reset_index()
product_desc.columns = ["stock_code", "description"]
product_desc["description"] = product_desc["description"].fillna("UNKNOWN")
product_desc["category"] = product_desc["description"].apply(assign_category)
product_desc["product_key"] = range(1, len(product_desc) + 1)

# Add the Unknown Product placeholder row
unknown_product = pd.DataFrame([{"product_key": -1, "stock_code": None, "description": "UNKNOWN", "category": "Other"}])
dim_product = pd.concat([unknown_product, product_desc[["product_key", "stock_code", "description", "category"]]], ignore_index=True)

print(f"dim_product rows: {len(dim_product)}")
print(dim_product["category"].value_counts())

dim_product rows: 4933
category
Other                   2337
Kitchen & Dining         620
Lighting                 496
Home Decor               494
Bags & Storage           471
Stationery & Paper       265
Christmas & Seasonal     250
Name: count, dtype: int64


### Build dim_customer (with the Unknown Customer placeholder)

In [7]:
customer_info = df[df["customer_id_raw"].notna()].groupby("customer_id_raw").agg(
    country=("country", "first"),
    first_purchase_date=("invoicedate", "min")
).reset_index()
customer_info.columns = ["customer_id", "country", "first_purchase_date"]
customer_info["customer_key"] = range(1, len(customer_info) + 1)
customer_info["customer_segment"] = None  # populated in Module 4

unknown_customer = pd.DataFrame([{
    "customer_key": -1, "customer_id": None, "country": None,
    "first_purchase_date": None, "customer_segment": None
}])

dim_customer = pd.concat([unknown_customer, customer_info[["customer_key", "customer_id", "country", "first_purchase_date", "customer_segment"]]], ignore_index=True)
print(f"dim_customer rows: {len(dim_customer)}")
dim_customer.head()

dim_customer rows: 5940


,customer_key,customer_id,country,first_purchase_date,customer_segment
0,-1,None,None,None,None
1,1,12346.0,United Kingdom,2009-12-14 08:34:00,None
2,2,12347.0,Iceland,2010-10-31 14:20:00,None
3,3,12348.0,Finland,2010-09-27 14:59:00,None
4,4,12349.0,Italy,2009-12-04 12:49:00,None


### Build fact_sales, joining everything via lookup maps

In [8]:
# Lookup maps: natural key -> surrogate key
country_map = dict(zip(dim_country["country_name"], dim_country["country_key"]))
product_map = dict(zip(dim_product["stock_code"], dim_product["product_key"]))
customer_map = dict(zip(dim_customer["customer_id"], dim_customer["customer_key"]))

fact_sales = df.copy()
fact_sales["date_key"] = fact_sales["invoicedate"].dt.strftime("%Y%m%d").astype(int)
fact_sales["country_key"] = fact_sales["country"].map(country_map)
fact_sales["product_key"] = fact_sales["stockcode"].map(product_map).fillna(-1).astype(int)
fact_sales["customer_key"] = fact_sales["customer_id_raw"].map(customer_map).fillna(-1).astype(int)
fact_sales["total_amount"] = fact_sales["quantity"] * fact_sales["price"]

fact_sales = fact_sales.rename(columns={"invoice": "invoice_number", "price": "unit_price"})
fact_sales = fact_sales[[
    "date_key", "customer_key", "product_key", "country_key",
    "invoice_number", "quantity", "unit_price", "total_amount", "is_return"
]].reset_index(drop=True)
fact_sales.insert(0, "sale_id", range(1, len(fact_sales) + 1))

print(f"fact_sales rows: {len(fact_sales):,}")
fact_sales.head()

fact_sales rows: 1,061,164


,sale_id,date_key,customer_key,product_key,country_key,invoice_number,quantity,unit_price,total_amount,is_return
0,1,20091201,740,4231,1,489434,12,6.95,83.4,False
1,2,20091201,740,3458,1,489434,12,6.75,81.0,False
2,3,20091201,740,3460,1,489434,12,6.75,81.0,False
3,4,20091201,740,1290,1,489434,48,2.10,100.8,False
4,5,20091201,740,635,1,489434,24,1.25,30.0,False


### Data quality check before loading (never skip this)

In [9]:
print("Data quality checks:")
print(f"  Nulls in date_key: {fact_sales['date_key'].isna().sum()}")
print(f"  Nulls in customer_key: {fact_sales['customer_key'].isna().sum()}")
print(f"  Nulls in product_key: {fact_sales['product_key'].isna().sum()}")
print(f"  Nulls in country_key: {fact_sales['country_key'].isna().sum()}")
print(f"  Total revenue check: ${fact_sales[~fact_sales['is_return']]['total_amount'].sum():,.2f}")

Data quality checks:
  Nulls in date_key: 0
  Nulls in customer_key: 0
  Nulls in product_key: 0
  Nulls in country_key: 0
  Total revenue check: $20,972,968.14
